## Extract to txt

In [ ]:
import os 
import re
import docx
from pdfminer.high_level import extract_text as extract_pdf_text
from tqdm import tqdm

def read_txt(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def read_docx(file_path):
    return "\n".join([p.text for p in docx.Document(file_path).paragraphs])

def read_pdf(file_path):
    return extract_pdf_text(file_path)

def extract_text(file_path):
    file_path_lower = file_path.lower()
    if file_path_lower.endswith(".pdf"):
        return read_pdf(file_path)
    if file_path_lower.endswith(".docx"):
        return read_docx(file_path)
    if file_path_lower.endswith(".txt"):
        return read_txt(file_path)

def clean_text(text):
    return re.sub(r"\s+", " ", re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)).strip()

def extract_all_texts(input_folder, output_folder):
    supported_extensions = (".pdf", ".docx", ".txt")
    for filename in tqdm(os.listdir(input_folder), desc="Extracting files"):
        if not filename.lower().endswith(supported_extensions):
            continue
        input_path = os.path.join(input_folder, filename)
        output_filename = f"{os.path.splitext(filename)[0]}.txt"
        output_path = os.path.join(output_folder, output_filename)
        if os.path.exists(output_path):
            continue
        text = clean_text(extract_text(input_path))
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)

extract_all_texts("./exploration/resume", "./exploration/resume_extract")

Extracting files: 100%|██████████| 2549/2549 [00:00<00:00, 14671.25it/s]


## Txt to Json

In [52]:
import os
import json
import json5
from time import sleep
from openai import OpenAI
from tqdm import tqdm

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key='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')

def safe_json_parse(content):
    try:
        return json5.loads(content)
    except Exception as e:
        print("⚠️ Malformed JSON:", e)
        return None

def parse_resume_with_llm(text):
    prompt = f"""
    You are a CV parsing system.
    Extract the following fields from the CV below and return STRICTLY valid JSON:
    - name
    - email
    - phone
    - location
    - summary
    - skills (list)
    - experience (list of objects: title, company, start_date, end_date, description)
    - education (list of objects: degree, school, year)
    - certifications (list)
    - languages (list)

    CV:
    --------------
    {text}
    --------------
    
    Reply ONLY with valid JSON.
    """
    response = client.chat.completions.create(
        model="openai/gpt-5-nano",
        messages=[{"role": "user", "content": prompt}]
    )
    return safe_json_parse(response.choices[0].message.content)

def parse_single_resume(path, max_retries=10):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read().strip()

    for _ in range(1, max_retries + 1):
        parsed = parse_resume_with_llm(text)
        if parsed:
            return parsed
        sleep(1)

    print(f"⛔ Échec du parsing après {max_retries} tentatives : {path}")
    return None

def process_resumes(input_folder, output_folder):
    txt_files = [f for f in os.listdir(input_folder) if f.lower().endswith(".txt")]

    for filename in tqdm(txt_files, desc="Processing CVs", unit="CV"):
        input_path = os.path.join(input_folder, filename)
        output_name = os.path.splitext(filename)[0] + ".json"
        output_path = os.path.join(output_folder, output_name)
        
        if os.path.exists(output_path):
            continue

        parsed = parse_single_resume(input_path)
        if not parsed:
            continue

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, indent=4, ensure_ascii=False)

In [ ]:
process_resumes("./exploration/resume_extract", "./exploration/resume_extract_json")

Processing CVs: 100%|██████████| 2549/2549 [00:00<00:00, 19343.66CV/s]


## 3. resume extract and conversion from json to text

In [ ]:
path_lkdn = "./exploration/linkedin_job_postings_cleaned.csv"

In [23]:
import os
import json
from tqdm import tqdm

### Conversion without experiences description

In [24]:
def clean_string(s):
    return str(s).replace("\n", " ").strip()

In [25]:
def json_to_text(data):
    parts = []

    skills = data.get("skills")
    if skills:
        skills_list = [clean_string(s) for s in (skills if isinstance(skills, list) else [skills]) if s]
        parts.append("Skills: " + "; ".join(skills_list))

    experience = data.get("experience")
    if experience:
        exp_list = []
        for e in experience:
            title = clean_string(e.get("title", ""))
            company = clean_string(e.get("company", ""))
            years = clean_string(e.get("years", ""))
            exp_list.append(" ".join(filter(None, [title, "at" if title and company else "", company, years])))
        parts.append("Experience: " + "; ".join(exp_list))

    education = data.get("education")
    if education:
        edu_list = []
        for e in education:
            degree = clean_string(e.get("degree", ""))
            school = clean_string(e.get("school", ""))
            edu_list.append(" ".join(filter(None, [degree, "at" if degree and school else "", school])))
        parts.append("Education: " + "; ".join(edu_list))

    certifications = data.get("certifications")
    if certifications:
        cert_list = [clean_string(c) for c in (certifications if isinstance(certifications, list) else [certifications]) if c]
        parts.append("Certifications: " + "; ".join(cert_list))

    summary = data.get("summary")
    if summary:
        parts.append("Summary: " + clean_string(summary))

    return "\n".join(parts).strip()

In [26]:
def process_json_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    for filename in tqdm([f for f in os.listdir(input_folder) if f.endswith(".json")], desc="Processing JSON files"):
        input_path = os.path.join(input_folder, filename)
        with open(input_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        output_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}.txt")
        with open(output_path, "w", encoding="utf-8") as out:
            out.write(json_to_text(data))

### Conversion with experiences description

In [27]:
def json_to_text1(data):
    parts = []
    skills = data.get("skills")
    if skills:
        skills_list = [clean_string(s) for s in (skills if isinstance(skills, list) else [skills]) if s]
        parts.append("Skills: " + "; ".join(skills_list))

    experience = data.get("experience")
    if experience:
        exp_list = []
        for e in experience:
            title = clean_string(e.get("title", ""))
            company = clean_string(e.get("company", ""))
            start_date = clean_string(e.get("start_date", ""))
            end_date = clean_string(e.get("end_date", ""))
            description = clean_string(e.get("description", ""))
            
            exp_parts = []
            if title:
                exp_parts.append(title)
            if company:
                exp_parts.append(f"at {company}")
            if start_date or end_date:
                date_range = f"({start_date} - {end_date})".replace("  ", " ").strip()
                exp_parts.append(date_range)
            if description:
                exp_parts.append(f": {description}")
            
            exp_list.append(" ".join(exp_parts))
        
        parts.append("Experience: " + " | ".join(exp_list))

    education = data.get("education")
    if education:
        edu_list = []
        for e in education:
            degree = clean_string(e.get("degree", ""))
            school = clean_string(e.get("school", ""))
            edu_list.append(" ".join(filter(None, [degree, "at" if degree and school else "", school])))
        parts.append("Education: " + "; ".join(edu_list))

    certifications = data.get("certifications")
    if certifications:
        cert_list = [clean_string(c) for c in (certifications if isinstance(certifications, list) else [certifications]) if c]
        if cert_list:
            parts.append("Certifications: " + "; ".join(cert_list))

    summary = data.get("summary")
    if summary:
        parts.append("Summary: " + clean_string(summary))

    return "\n".join(parts).strip()


In [28]:
def process_json_folder1(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    for filename in tqdm([f for f in os.listdir(input_folder) if f.endswith(".json")], desc="Processing JSON files"):
        input_path = os.path.join(input_folder, filename)
        with open(input_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        output_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}.txt")
        with open(output_path, "w", encoding="utf-8") as out:
            out.write(json_to_text1(data))

In [ ]:
# first approach
process_json_folder("./exploration/resume_extract_json", "./exploration/resume_extract_text")

Processing JSON files: 100%|██████████| 2549/2549 [00:09<00:00, 259.51it/s]


In [ ]:
# second approach
process_json_folder1("./exploration/resume_extract_json", "./exploration/resume_extract_text1")

Processing JSON files: 100%|██████████| 2549/2549 [00:09<00:00, 269.60it/s]


## 4. resume embedding

### Embedding with smaller extracted text

In [10]:
import os
import json
import numpy as np
import faiss
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.utils import simple_preprocess

c:\Users\sosol\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Embedding with SentenceTransformer

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2") #"all-mpnet-base-v2"

input_folder = "./exploration/resume_extract_text"
txt_files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]

batch_size = 32
embeddings_list = []

for start in tqdm(range(0, len(txt_files), batch_size), desc="Encoding resumes"):
    batch_files = txt_files[start:start + batch_size]
    texts = []
    for file in batch_files:
        path = os.path.join(input_folder, file)
        try:
            with open(path, "r", encoding="utf-8") as f:
                texts.append(f.read())
        except Exception as e:
            print(f"Error reading {file}: {e}")
            texts.append("")

    batch_embeddings = model.encode(texts, convert_to_numpy=True, batch_size=batch_size)
    batch_embeddings /= np.linalg.norm(batch_embeddings, axis=1, keepdims=True)
    embeddings_list.append(batch_embeddings)

all_embeddings = np.vstack(embeddings_list)
faiss_index = faiss.IndexFlatIP(all_embeddings.shape[1])
faiss_index.add(all_embeddings)
faiss.write_index(faiss_index, "./exploration/resume_index.faiss")

with open("./exploration/resume_index_mapping.json", "w", encoding="utf-8") as f:
    json.dump(txt_files, f, ensure_ascii=False, indent=4)

c:\Users\sosol\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Encoding resumes: 100%|██████████| 80/80 [00:08<00:00,  9.59it/s]


### Embedding with bigger extracted text

Embedding with SentenceTransformer

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2") #"all-mpnet-base-v2"

input_folder = "./exploration/resume_extract_text1"
txt_files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]

batch_size = 32
embeddings_list = []

for start in tqdm(range(0, len(txt_files), batch_size), desc="Encoding resumes"):
    batch_files = txt_files[start:start + batch_size]
    texts = []
    for file in batch_files:
        path = os.path.join(input_folder, file)
        try:
            with open(path, "r", encoding="utf-8") as f:
                texts.append(f.read())
        except Exception as e:
            print(f"Error reading {file}: {e}")
            texts.append("")

    batch_embeddings = model.encode(texts, convert_to_numpy=True, batch_size=batch_size)
    batch_embeddings /= np.linalg.norm(batch_embeddings, axis=1, keepdims=True)
    embeddings_list.append(batch_embeddings)

all_embeddings = np.vstack(embeddings_list)
faiss_index = faiss.IndexFlatIP(all_embeddings.shape[1])
faiss_index.add(all_embeddings)
faiss.write_index(faiss_index, "./exploration/resume_index1.faiss")

with open("./exploration/resume_index_mapping1.json", "w", encoding="utf-8") as f:
    json.dump(txt_files, f, ensure_ascii=False, indent=4)

c:\Users\sosol\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Encoding resumes: 100%|██████████| 80/80 [00:06<00:00, 11.96it/s]


## 5. job offer embedding

In [15]:
import pandas as pd
import torch

In [16]:

print("Path to dataset files:", path_lkdn)
df = pd.read_csv(path_lkdn)
df = df.sample(650000)

fields_to_combine = ["job_link", "job_title", "company", "job_location", "search_city","job_type", "search_position", "job_skills", "job_summary"]
df["combined_text"] = df[fields_to_combine].astype(str).agg(" ".join, axis=1)

Path to dataset files: C:\Users\sosol\Documents\Travail\DataIA\GenAI\projet\Jobs_Linkedin\linkedin_job_postings_cleaned.csv


### Embedding with SentenceTransformer

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
job_embeddings = model.encode(df["combined_text"].tolist(), convert_to_numpy=True, show_progress_bar=True, normalize_embeddings=True)

faiss_index = faiss.IndexFlatIP(job_embeddings.shape[1])
faiss_index.add(job_embeddings)
faiss.write_index(faiss_index, "./exploration/jobs_index.faiss")

job_id_mapping = {i: jid for i, jid in enumerate(df["job_link"].tolist())}
with open("./exploration/jobs_index_mapping.json", "w", encoding="utf-8") as f:
    json.dump(job_id_mapping, f, indent=4)

c:\Users\sosol\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Batches:   0%|          | 0/20313 [00:00<?, ?it/s]

## 6. resume and job offer match

In [29]:
import umap.umap_ as umap
import matplotlib.pyplot as plt

In [18]:
CV_INDEX = [np.random.choice(2248) for _ in range(50)]
TOP_N = 2   

### Matching with smaller embedded resume text

Matching with SentenceTransformer embedding

In [ ]:
jobs_index = faiss.read_index("./exploration/jobs_index.faiss")
with open("./exploration/jobs_index_mapping.json", "r") as f:
    jobs_mapping = {int(k): v for k, v in json.load(f).items()}

resume_index = faiss.read_index("./exploration/resume_index.faiss")
with open("./exploration/resume_index_mapping.json", "r") as f:
    resume_mapping = json.load(f)

model = SentenceTransformer("all-MiniLM-L6-v2")

def reconstruct_embeddings(index):
    embeddings = np.zeros((index.ntotal, index.d), dtype=np.float32)
    for i in range(index.ntotal):
        index.reconstruct(i, embeddings[i])
    return embeddings

resume_json_folder = "./exploration/resume_extract_text"
resume_data = {}

for fname in os.listdir(resume_json_folder):
    if fname.lower().endswith(".txt"):
        path = os.path.join(resume_json_folder, fname)
        with open(path, "r", encoding="utf-8") as f:
            data = f.read()
        resume_data[fname] = data

jobs_df = pd.read_csv(path_lkdn)

cv_embeddings = reconstruct_embeddings(resume_index)
job_embeddings = reconstruct_embeddings(jobs_index)
all_embeddings = np.vstack([cv_embeddings, job_embeddings])

# Store all similarity scores
all_similarity_scores = []

for cv_index in CV_INDEX:
    cv_filename = resume_mapping[cv_index]

    print("===== Selected resume =====")
    if cv_filename in resume_data:
        print(resume_data[cv_filename])

    cv_emb = cv_embeddings[cv_index:cv_index+1]
    distances, indices = jobs_index.search(cv_emb, TOP_N)

    results = [
        {"job_index": int(idx), "job_link": jobs_mapping[int(idx)], "score": float(score)}
        for score, idx in zip(distances[0], indices[0])
    ]

    cv_scores = [r["score"] for r in results]
    all_similarity_scores.extend(cv_scores)

    print(f"\n===== Top {TOP_N} matching job offers =====")

    for r in results:
        job_link = r["job_link"]
        
        print(f"\n--- 💼 Found offers : {job_link} ---")
        print(f"Similarity score : {r['score']:.4f}")
        
        job_row = jobs_df[jobs_df["job_link"] == job_link]

        if len(job_row) == 0:
            print("⚠️ Offer not found in jobs_df")
            continue
        
        row = job_row.iloc[0]
        
        fields_to_show = [
            "job_link", "company", "job_location", "job_title", 
            "job_skills", "job_summary", "search_city", "search_country", "job_type"
        ]
        
        for col in fields_to_show:
            if col in row and not pd.isna(row[col]):
                print(f"{col}: {row[col]}")

# Calculate and display statistics at the end
print("\n")
print("📊 SIMILARITY SCORE STATISTICS")
print("="*50)
if all_similarity_scores:
    average_score = np.mean(all_similarity_scores)
    median_score = np.median(all_similarity_scores)

    print(f"Total matches evaluated: {len(all_similarity_scores)}")
    print(f"Average similarity score: {average_score:.4f}")
    print(f"Median similarity score: {median_score:.4f}")
else:
    print("No similarity scores were computed.")
print("="*50)

c:\Users\sosol\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


===== Selected resume =====
Skills: blueprints; Read blueprints; interpersonal & communication; conferences; customer relations; excellent customer service; direction; Hiring; Layout; materials; meetings; networking; new construction; personnel; Plumbing; plumber; improve process; progress; quality; quality control; repairs; Research; Safety; FM; scheduling; seminars; tear; technical assistance; Troubleshooting
Experience: Sales at Sales Company Name; Broadcast Engineer / Construction Project Manager at Unknown; Plumber at The Plumbing Company
Education: High School Diploma at Somerville High School; Associate of Arts at Somerset County College; Bachelor of Arts at Geneva College
Summary: Talented Construction Manager with over twenty years of experience as an Independent Contractor. Experienced in managing projects of all sizes, cost estimation, and delivering on time and under budget. Strong leadership, communication, and relationship-building skills. Able to manage multiple tasks un

### Matching with bigger embedded resume text

Matching with SentenceTransformer embedding

In [ ]:
jobs_index = faiss.read_index("./exploration/jobs_index.faiss")
with open("./exploration/jobs_index_mapping.json", "r") as f:
    jobs_mapping = {int(k): v for k, v in json.load(f).items()}

resume_index = faiss.read_index("./exploration/resume_index1.faiss")
with open("./exploration/resume_index_mapping1.json", "r") as f:
    resume_mapping = json.load(f)

model = SentenceTransformer("all-MiniLM-L6-v2")

def reconstruct_embeddings(index):
    embeddings = np.zeros((index.ntotal, index.d), dtype=np.float32)
    for i in range(index.ntotal):
        index.reconstruct(i, embeddings[i])
    return embeddings

resume_json_folder = "./exploration/resume_extract_text1"
resume_data = {}

for fname in os.listdir(resume_json_folder):
    if fname.lower().endswith(".txt"):
        path = os.path.join(resume_json_folder, fname)
        with open(path, "r", encoding="utf-8") as f:
            data = f.read()
        resume_data[fname] = data

jobs_df = pd.read_csv(path_lkdn)

cv_embeddings = reconstruct_embeddings(resume_index)
job_embeddings = reconstruct_embeddings(jobs_index)
all_embeddings = np.vstack([cv_embeddings, job_embeddings])

# Store all similarity scores
all_similarity_scores = []

for cv_index in CV_INDEX:
    cv_filename = resume_mapping[cv_index]

    print("===== Selected resume =====")
    if cv_filename in resume_data:
        print(resume_data[cv_filename])

    cv_emb = cv_embeddings[cv_index:cv_index+1]
    distances, indices = jobs_index.search(cv_emb, TOP_N)

    results = [
        {"job_index": int(idx), "job_link": jobs_mapping[int(idx)], "score": float(score)}
        for score, idx in zip(distances[0], indices[0])
    ]

    cv_scores = [r["score"] for r in results]
    all_similarity_scores.extend(cv_scores)

    print(f"\n===== Top {TOP_N} matching job offers =====")

    for r in results:
        job_link = r["job_link"]
        
        print(f"\n--- 💼 Found offers : {job_link} ---")
        print(f"Similarity score : {r['score']:.4f}")
        
        job_row = jobs_df[jobs_df["job_link"] == job_link]

        if len(job_row) == 0:
            print("⚠️ Offer not found in jobs_df")
            continue
        
        row = job_row.iloc[0]
        
        fields_to_show = [
            "job_link", "company", "job_location", "job_title", 
            "job_skills", "job_summary", "search_city", "search_country", "job_type"
        ]
        
        for col in fields_to_show:
            if col in row and not pd.isna(row[col]):
                print(f"{col}: {row[col]}")

# Calculate and display statistics at the end
print("\n")
print("📊 SIMILARITY SCORE STATISTICS")
print("="*50)
if all_similarity_scores:
    average_score = np.mean(all_similarity_scores)
    median_score = np.median(all_similarity_scores)

    print(f"Total matches evaluated: {len(all_similarity_scores)}")
    print(f"Average similarity score: {average_score:.4f}")
    print(f"Median similarity score: {median_score:.4f}")
else:
    print("No similarity scores were computed.")
print("="*50)

c:\Users\sosol\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


===== Selected resume =====
Skills: blueprints; Read blueprints; interpersonal & communication; conferences; customer relations; excellent customer service; direction; Hiring; Layout; materials; meetings; networking; new construction; personnel; Plumbing; plumber; improve process; progress; quality; quality control; repairs; Research; Safety; FM; scheduling; seminars; tear; technical assistance; Troubleshooting
Experience: Sales at Sales Company Name; Broadcast Engineer / Construction Project Manager at Unknown; Plumber at The Plumbing Company
Education: High School Diploma at Somerville High School; Associate of Arts at Somerset County College; Bachelor of Arts at Geneva College
Summary: Talented Construction Manager with over twenty years of experience as an Independent Contractor. Experienced in managing projects of all sizes, cost estimation, and delivering on time and under budget. Strong leadership, communication, and relationship-building skills. Able to manage multiple tasks un

## Testing pipeline for matching job offer retrieval

In [5]:
import os
import docx
from pdfminer.high_level import extract_text as extract_pdf_text
import re
import json5
from time import sleep
from openai import OpenAI

In [ ]:
def read_txt(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def read_docx(file_path):
    return "\n".join([p.text for p in docx.Document(file_path).paragraphs])

def read_pdf(file_path):
    return extract_pdf_text(file_path)

def extract_text(file_path):
    file_path_lower = file_path.lower()
    if file_path_lower.endswith(".pdf"):
        return read_pdf(file_path)
    if file_path_lower.endswith(".docx"):
        return read_docx(file_path)
    if file_path_lower.endswith(".txt"):
        return read_txt(file_path)

def clean_text(text):
    return re.sub(r"\s+", " ", re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)).strip()

def extract_all_texts(input_folder, output_folder):
    supported_extensions = (".pdf", ".docx", ".txt")
    for filename in tqdm(os.listdir(input_folder), desc="Extracting files"):
        if not filename.lower().endswith(supported_extensions):
            continue
        input_path = os.path.join(input_folder, filename)
        output_filename = f"{os.path.splitext(filename)[0]}.txt"
        output_path = os.path.join(output_folder, output_filename)
        if os.path.exists(output_path):
            continue
        text = clean_text(extract_text(input_path))
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)
api_key = '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'
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

def safe_json_parse(content):
    try:
        return json5.loads(content)
    except Exception as e:
        print("⚠️ Malformed JSON:", e)
        return None

def parse_resume_with_llm(text):
    prompt = f"""
    You are a CV parsing system.
    Extract the following fields from the CV below and return STRICTLY valid JSON:
    - name
    - email
    - phone
    - location
    - summary
    - skills (list)
    - experience (list of objects: title, company, start_date, end_date, description)
    - education (list of objects: degree, school, year)
    - certifications (list)
    - languages (list)

    CV:
    --------------
    {text}
    --------------
    
    Reply ONLY with valid JSON.
    """
    response = client.chat.completions.create(
        model="openai/gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}]
    )
    return safe_json_parse(response.choices[0].message.content)

def parse_single_resume(path, max_retries=10):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read().strip()

    for _ in range(1, max_retries + 1):
        parsed = parse_resume_with_llm(text)
        if parsed:
            return parsed
        sleep(1)

    print(f"⛔ Échec du parsing après {max_retries} tentatives : {path}")
    return None

def process_resumes(input_folder, output_folder):
    txt_files = [f for f in os.listdir(input_folder) if f.lower().endswith(".txt")]
    os.makedirs(output_folder, exist_ok=True)
    for filename in tqdm(txt_files, desc="Processing CVs", unit="CV"):
        input_path = os.path.join(input_folder, filename)
        output_name = os.path.splitext(filename)[0] + ".json"
        output_path = os.path.join(output_folder, output_name)
        
        if os.path.exists(output_path):
            continue
        print(f"Processing CV: {filename}")
        parsed = parse_single_resume(input_path)
        if not parsed:
            continue

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, indent=4, ensure_ascii=False)
        print(f"File saved successfully: {output_path}")

def clean_CV(path_to_directory):
    os.makedirs(path_to_directory, exist_ok=True)
    extract_all_texts("./exploration/resume_test", path_to_directory)
    process_resumes(path_to_directory, "./exploration/resume_test_extract_json")

def json_to_text(data):
    parts = []

    skills = data.get("skills")
    if skills:
        skills_list = [clean_string(s) for s in (skills if isinstance(skills, list) else [skills]) if s]
        parts.append("Skills: " + "; ".join(skills_list))

    experience = data.get("experience")
    if experience:
        exp_list = []
        for e in experience:
            title = clean_string(e.get("title", ""))
            company = clean_string(e.get("company", ""))
            years = clean_string(e.get("years", ""))
            exp_list.append(" ".join(filter(None, [title, "at" if title and company else "", company, years])))
        parts.append("Experience: " + "; ".join(exp_list))

    education = data.get("education")
    if education:
        edu_list = []
        for e in education:
            degree = clean_string(e.get("degree", ""))
            school = clean_string(e.get("school", ""))
            edu_list.append(" ".join(filter(None, [degree, "at" if degree and school else "", school])))
        parts.append("Education: " + "; ".join(edu_list))

    certifications = data.get("certifications")
    if certifications:
        cert_list = [clean_string(c) for c in (certifications if isinstance(certifications, list) else [certifications]) if c]
        parts.append("Certifications: " + "; ".join(cert_list))

    summary = data.get("summary")
    if summary:
        parts.append("Summary: " + clean_string(summary))

    return "\n".join(parts).strip()   

def clean_string(s):
    return str(s).replace("\n", " ").strip()

In [ ]:
# Clean the CV and match jobs
clean_CV("./exploration/resume_test_extract_text")
cv = json_to_text(parse_single_resume("./exploration/resume_test_extract_json/SOLAL_LEDRU_EN.json"))

Processing CVs: 100%|██████████| 1/1 [00:00<?, ?CV/s]


In [40]:
jobs_df = pd.read_csv(path_lkdn)

In [ ]:
jobs_index = faiss.read_index("./exploration/jobs_index.faiss")
with open("./exploration/jobs_index_mapping.json", "r") as f:
    jobs_mapping = {int(k): v for k, v in json.load(f).items()}

model = SentenceTransformer("all-MiniLM-L6-v2")

def reconstruct_embeddings(index):
    embeddings = np.zeros((index.ntotal, index.d), dtype=np.float32)
    for i in range(index.ntotal):
        index.reconstruct(i, embeddings[i])
    return embeddings

def analyze_job_match(cv_text, job_details):
    """
    Uses LLM to analyze why the candidate is suited for the job
    """
    prompt = f"""
You are an expert career advisor and recruiter. Analyze the following CV and job offer to explain why this candidate is suited for this position.

**CANDIDATE CV:**
{cv_text[:3000]}  

**JOB OFFER:**
- Company: {job_details.get('company', 'N/A')}
- Title: {job_details.get('job_title', 'N/A')}
- Location: {job_details.get('job_location', 'N/A')}
- Job Type: {job_details.get('job_type', 'N/A')}
- Required Skills: {job_details.get('job_skills', 'N/A')}
- Job Summary: {job_details.get('job_summary', 'N/A')}

**TASK:**
Provide a concise analysis (150-200 words) covering:
1. **Key Strengths**: Which candidate skills/experiences match the job requirements?
2. **Alignment**: How does the candidate's background align with the role?
3. **Potential Gaps**: Are there any areas where the candidate might need development?
4. **Overall Fit**: Rate the fit as Excellent/Good/Fair/Poor and explain why.

Adress the candidate as "You"

Be specific, professional, and constructive.
"""
    
    try:
        response = client.chat.completions.create(
            model="openai/gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=400
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"❌ Error generating analysis: {e}"

jobs_df = pd.read_csv(path_lkdn)
job_embeddings = reconstruct_embeddings(jobs_index)

# Store all similarity scores
all_similarity_scores = []
all_analyses = []

# Generate embedding for the CV string
print("🔄 Generating embedding for the provided CV...")
cv_embedding = model.encode([cv], convert_to_numpy=True)

print("===== Selected resume =====")
print(cv[:500])  # Print first 500 characters
print("...\n")

# Search for matching jobs
distances, indices = jobs_index.search(cv_embedding, TOP_N)

results = [
    {"job_index": int(idx), "job_link": jobs_mapping[int(idx)], "score": float(score)}
    for score, idx in zip(distances[0], indices[0])
]

cv_scores = [r["score"] for r in results]
all_similarity_scores.extend(cv_scores)

print(f"\n===== Top {TOP_N} matching job offers =====\n")

for i, r in enumerate(results, 1):
    job_link = r["job_link"]

    print(f"\n{'='*80}")
    print(f"MATCH #{i} - 💼 {job_link}")
    print(f"{'='*80}")
    print(f"📊 Similarity score: {r['score']:.4f}")

    job_row = jobs_df[jobs_df["job_link"] == job_link]

    if len(job_row) == 0:
        print("⚠️ Offer not found in jobs_df")
        continue

    row = job_row.iloc[0]

    # Display job details
    fields_to_show = [
        "company", "job_title", "job_location", "job_type",
        "job_skills", "job_summary", "search_city", "search_country"
    ]

    print("\n📋 JOB DETAILS:")
    print("-" * 80)
    job_details = {}
    for col in fields_to_show:
        if col in row and not pd.isna(row[col]):
            value = row[col]
            job_details[col] = value
            # Format display
            display_col = col.replace('_', ' ').title()
            print(f"  {display_col}: {value}")
    
    job_details['job_link'] = job_link

    # Generate LLM analysis
    print("\n🤖 AI MATCH ANALYSIS:")
    print("-" * 80)
    analysis = analyze_job_match(cv, job_details)
    print(analysis)
    
    all_analyses.append({
        "job_link": job_link,
        "score": r['score'],
        "analysis": analysis
    })
    
    print(f"\n{'='*80}\n")

# Calculate and display statistics at the end
print("\n")
print("📊 SIMILARITY SCORE STATISTICS")
print("="*80)
if all_similarity_scores:
    average_score = np.mean(all_similarity_scores)
    median_score = np.median(all_similarity_scores)
    min_score = np.min(all_similarity_scores)
    max_score = np.max(all_similarity_scores)
    std_score = np.std(all_similarity_scores)

    print(f"Total matches evaluated: {len(all_similarity_scores)}")
    print(f"Average similarity score: {average_score:.4f}")
    print(f"Median similarity score: {median_score:.4f}")
    print(f"Min similarity score: {min_score:.4f}")
    print(f"Max similarity score: {max_score:.4f}")
    print(f"Standard deviation: {std_score:.4f}")
else:
    print("No similarity scores were computed.")
print("="*80)

# Optional: Save all analyses to a file
print("\n💾 Saving analysis report...")
report = {
    "cv_preview": cv[:500],
    "total_matches": len(all_analyses),
    "statistics": {
        "average_score": float(average_score) if all_similarity_scores else 0,
        "median_score": float(median_score) if all_similarity_scores else 0,
        "min_score": float(min_score) if all_similarity_scores else 0,
        "max_score": float(max_score) if all_similarity_scores else 0,
    },
    "matches": all_analyses
}

with open("./exploration/job_match_analysis.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4, ensure_ascii=False)

print("✅ Analysis saved to: ./exploration/job_match_analysis.json")

c:\Users\sosol\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


🔄 Generating embedding for the provided CV...
===== Selected resume =====
Skills: Python: AI & machine learning with TensorFlow, PyTorch & data preprocessing with numpy, pandas, sklearn; R: statistics & data manipulation and visualization; SQL: database management; GitHub: MLOps; Cloud: AWS; Microsoft 365; Power BI; Power Automate
Experience: Sailing instructor assistant at St Cyr sur Mer Sailing Club; General employee at Château Salettes; Intern in the Technical Department of Information Services at Orange; AI/Human Factors Cooperation Intern at Human Design Group & 
...


===== Top 2 matching job offers =====


MATCH #1 - 💼 https://www.linkedin.com/jobs/view/ai-practice-senior-engineer-at-unisys-3775483673
📊 Similarity score: 0.6696

📋 JOB DETAILS:
--------------------------------------------------------------------------------
  Company: Unisys
  Job Title: AI Practice- Senior Engineer
  Job Location: United, PA
  Job Type: Onsite
  Job Skills: AI, Machine Learning, Generative AI, L